# Day 09 — Async Python
Vidya Sankalp | Applied GenAI Engineering

> **The arc:** plain generators (Day 03) -> async generators (today) -> SSE streaming in FastAPI (Day 10). By the end you'll read your project's streaming code without anything feeling like magic.

> **9 sections:** foundations | core mechanics | running concurrently | control & resilience | bridging blocking code | async generators | the Queue pattern (the A2A bridge) | safe shared state | going further.

> **Project mapping:** every deep concept is annotated with the real file it appears in - `agent_health.py`, `chat.py`, `supervisor_client.py`, `pg_client.py`, and the `EventQueue` inside every agent's `execute()`.

> **Jupyter note:** a notebook already runs an event loop, so cells use **top-level `await`** directly. In a real `.py` script you wrap the entry point in `asyncio.run(main())` (shown in section 2).


---


## 1 - Foundations: Why Async Exists

### Concurrency vs parallelism (one line each)
- **Parallelism** = doing many things *at the same instant* (multiple CPU cores grinding numbers).
- **Concurrency** = *managing* many things in progress, switching between them while they wait. Async is concurrency, not parallelism.

### I/O-bound vs CPU-bound
| Kind | Example | Does async help? |
|------|---------|------------------|
| **I/O-bound** | waiting on a database, API, network, disk | **Yes** - the program is just *waiting* |
| **CPU-bound** | resizing images, training a model, heavy math | **No** - needs multiprocessing |

Your project is overwhelmingly **I/O-bound**: it waits on Postgres, DynamoDB, 8 agent services, and the LLM. That waiting is exactly what async turns into overlap.


In [ ]:
import asyncio, time

# 'Blocking' means: this line refuses to let anything else run until it's done.
def check_service_sync(name):
    time.sleep(1)                  # BLOCKS - the whole program freezes here
    return f'{name}: healthy'

t0 = time.perf_counter()
results = [check_service_sync(s) for s in ['sql', 'chart', 'export']]
print(results)
print(f'Sequential: {time.perf_counter()-t0:.1f}s  (3 blocking waits, back to back)')


Three one-second waits took three seconds, and the CPU did *nothing* - it just waited. That idle waiting is the waste async removes. We fix it in section 3, but first the grammar.


---


## 2 - Core Mechanics: `async def`, `await`, the Event Loop

| Term | Meaning |
|------|---------|
| **coroutine** | a function defined with `async def` - it can pause and resume |
| **`await`** | "pause here until this finishes; let other coroutines run while I wait" |
| **event loop** | the scheduler that runs many coroutines, switching at each `await` |
| **`asyncio.run()`** | starts the loop in a `.py` file (Jupyter already has one) |

The key surprise: **calling an `async def` function does not run it.** It hands you a *coroutine object*. Only `await` actually runs it.


In [ ]:
async def get_price(product):
    await asyncio.sleep(0.2)              # 'await' = pause here, let others run
    prices = {'monitor': 205.21, 'mouse': 29.99}
    return prices.get(product, 0.0)

coro = get_price('monitor')              # NOT run yet - just a coroutine object
print('type:', type(coro).__name__)
price = await get_price('monitor')       # await actually runs it
print('price:', price)


In [ ]:
# Awaiting in SEQUENCE is correct but NOT concurrent.
t0 = time.perf_counter()
a = await get_price('monitor')   # 0.2s ...
b = await get_price('mouse')     # ... then another 0.2s
print(a, b, f'-> {time.perf_counter()-t0:.1f}s (sequential awaits)')


In [ ]:
# In a .py FILE you start the loop with asyncio.run(). (Reference - don't run in Jupyter.)
reference = '''
import asyncio
async def main():
    print(await get_price("monitor"))
if __name__ == "__main__":
    asyncio.run(main())     # creates loop, runs main(), closes loop
'''
print(reference)


---


## 3 - Running Concurrently: `gather()` and `create_task()`

### `asyncio.gather()` - fan-out, ordered results
Runs several coroutines **all at once** and returns results **in the order you passed them**. This is how the platform health-checks all 8 agents in one shot.

> **Project:** `platform/services/agent_health.py` - pings every agent with `asyncio.gather(*[...])`.


In [ ]:
async def ping_agent(name):
    await asyncio.sleep(0.5)                    # network round-trip
    return {'name': name, 'status': 'healthy'}

async def check_all_agents(names):
    tasks = [ping_agent(n) for n in names]      # one coroutine per agent
    return await asyncio.gather(*tasks)         # * unpacks list into arguments

agents = ['sql', 'chart', 'export', 'web_search', 'supervisor']
t0 = time.perf_counter()
health = await check_all_agents(agents)
for h in health: print(h)
print(f'Checked {len(agents)} agents in {time.perf_counter()-t0:.1f}s (not {len(agents)*0.5:.1f}s)')


Five pings, half a second each, **0.5s total**. Sequentially this is 2.5s.

### `asyncio.create_task()` - start now, don't wait (background work)
`gather` waits for everything. `create_task` schedules a coroutine in the background and lets you keep going.

> **Project:** `platform/routers/chat.py` - `asyncio.create_task(_persist_user_message(...))` saves without making the user wait.


In [ ]:
saved = []

async def save_to_db(message):
    await asyncio.sleep(0.3)
    saved.append(message)
    print(f'  (background) saved: {message!r}')

async def handle_request(text):
    task = asyncio.create_task(save_to_db(text))   # starts immediately, in background
    print('Replied to user right away (not waiting for the save)')
    await asyncio.sleep(0.1)                        # ... do other work ...
    await task                                      # await before finishing
    print('Save confirmed.')

await handle_request('Where is my order #3042?')
print('DB now holds:', saved)


### Tasks vs coroutines
| | Coroutine | Task |
|-|-----------|------|
| created by | `get_price(...)` | `asyncio.create_task(get_price(...))` |
| running yet? | **No** - inert until awaited | **Yes** - scheduled immediately |
| use when | you'll `await` it now | you want it running in the background |

> **Gotcha:** keep a reference (`task = asyncio.create_task(...)`) or Python may garbage-collect it mid-run; `await` it eventually so errors aren't lost.


---


## 4 - Control & Resilience: Timeouts and Surviving Failures

### `asyncio.wait_for()` - bound a slow call
Gives a coroutine a time budget. Exceed it -> `TimeoutError` to handle, instead of waiting forever.

> **Project:** `title_generator.py` + `chat.py` cap title generation at 5s so a slow LLM never blocks the chat.


In [ ]:
async def generate_title(text):
    await asyncio.sleep(2)                  # pretend the LLM is slow today
    return f'Title: {text[:20]}'

try:
    title = await asyncio.wait_for(generate_title('quarterly revenue'), timeout=0.5)
    print('Title:', title)
except asyncio.TimeoutError:
    title = 'Untitled conversation'         # safe fallback
    print('Timed out -> fallback:', title)


### `return_exceptions=True` - one failure shouldn't sink the batch
By default, if one coroutine in `gather()` raises, the whole call fails. `return_exceptions=True` returns the exception **as a value** so the batch survives.

> **Project:** the research fan-out in `export_agent/research/research_agent.py`.


In [ ]:
async def fetch_review(review_id):
    await asyncio.sleep(0.2)
    if review_id == 999:
        raise ValueError(f'review {review_id} not found')   # this one fails
    return {'review_id': review_id, 'rating': 5}

results = await asyncio.gather(
    fetch_review(101), fetch_review(999), fetch_review(103),
    return_exceptions=True,                 # failures come back as values
)
for r in results:
    if isinstance(r, Exception): print('FAILED:', r)
    else:                        print('OK:    ', r)


---


## 5 - Bridging Blocking Code: `asyncio.to_thread()`

Many libraries (boto3/DynamoDB, `requests`) are **blocking** - calling them directly freezes the event loop. `asyncio.to_thread()` runs the blocking call in a separate thread so the loop keeps spinning.

> **Project:** used **62 times** - every DynamoDB call is `await asyncio.to_thread(save_message, ...)`.


In [ ]:
def save_message_blocking(text):     # regular def, blocking I/O
    time.sleep(0.3)                  # e.g. a sync DynamoDB write
    return f'saved: {text}'

async def handle():
    # WRONG: save_message_blocking('hi')   -> freezes the loop for 0.3s
    # RIGHT: hand it to a thread, await the result
    result = await asyncio.to_thread(save_message_blocking, 'hi')
    print(result)

await handle()


| The library is... | Use |
|-------------------|-----|
| async-native (`asyncpg`, `httpx.AsyncClient`) | `await the_async_call(...)` |
| blocking/sync (`boto3`, `requests`, `time.sleep`) | `await asyncio.to_thread(the_sync_call, ...)` |


---


## 6 - Async Generators: `async for` + `yield`

**Recall plain generators from Day 03:** a normal function `return`s once; a *generator* `yield`s many values, one at a time, pausing in between.

```python
# Day 03 - plain (sync) generator
def countdown(n):
    while n > 0:
        yield n          # produce a value, pause, resume on next request
        n -= 1
```

An **async generator** is the same idea plus one new thing: it can `await` between yields. That's the only new concept - `yield` you already know.

> **Project:** `supervisor_client.py` `stream_supervisor()` `yield`s events token-by-token (lines 323-355); the chat route consumes it with `async for`.


In [ ]:
async def stream_tokens(sentence):
    for word in sentence.split():
        await asyncio.sleep(0.15)        # tokens arrive over time (the await)
        yield word                        # hand each one out as it's ready

collected = []
async for token in stream_tokens('Electronics leads revenue this quarter'):
    print(token, end=' ', flush=True)     # appears progressively
    collected.append(token)
print()
print('Full message:', ' '.join(collected))


In [ ]:
# Mirrors supervisor_client.py: yield structured EVENTS, not just words.
import json

async def stream_supervisor(question):
    yield json.dumps({'type': 'reasoning', 'text': '[supervisor] routing to sql_agent'})
    answer = f'Electronics leads at $1.24M for: {question}'
    for word in answer.split():
        await asyncio.sleep(0.05)
        yield json.dumps({'type': 'text', 'text': word + ' '})
    yield json.dumps({'type': 'done'})

async for event in stream_supervisor('top category this quarter'):
    print(event)


On Day 10 the FastAPI route wraps each yielded event as `data: {...}` and hands the generator to `StreamingResponse` - that's the whole SSE pipeline.


---


## 7 - The Queue Pattern: Producer -> Queue -> Consumer (the A2A bridge)

This section unlocks your *own* agent code. Every agent in the project has this signature:

```python
async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
    await event_queue.enqueue_event(task)      # the PRODUCER side
```

That `EventQueue` is an async queue. Your agent is a **producer** - it `enqueue`s events. The A2A framework is the **consumer** - it drains the queue and streams each event to the browser as SSE. Understand `asyncio.Queue` and you understand `EventQueue`.

- `await queue.put(item)` - producer adds an item
- `await queue.get()` - consumer takes the next item (waits if empty)


In [ ]:
DONE = object()   # a sentinel value meaning 'no more items'

async def agent_execute(question, event_queue):
    """PRODUCER - mirrors your agent's execute(self, context, event_queue)."""
    await event_queue.put({'type': 'status', 'text': 'received question'})
    await asyncio.sleep(0.2)
    await event_queue.put({'type': 'status', 'text': 'querying database'})
    await asyncio.sleep(0.2)
    await event_queue.put({'type': 'result', 'text': 'Electronics: $1.24M'})
    await event_queue.put(DONE)        # signal the consumer we're finished

async def consumer(event_queue):
    """CONSUMER - mirrors the A2A framework draining the queue to SSE."""
    while True:
        event = await event_queue.get()    # waits if the queue is empty
        if event is DONE: break
        print('  -> SSE:', event)

queue = asyncio.Queue()
await asyncio.gather(
    agent_execute('top category?', queue),
    consumer(queue),
)


Producer and consumer run **at the same time**, connected only by the queue - the producer never waits for the consumer. Swap `asyncio.Queue` for the A2A `EventQueue` and `put` for `enqueue_event`, and this *is* your agent.

### `asyncio.Event` - a one-shot 'something happened' signal
Where a Queue passes *data*, an Event passes a *signal*: one coroutine waits, another flips it on. It's the idea behind 'task is now ready / working / done' status flips.


In [ ]:
ready = asyncio.Event()

async def waiter():
    print('waiter: blocked until ready is set')
    await ready.wait()                  # pauses until someone calls ready.set()
    print('waiter: got the signal, proceeding')

async def setter():
    await asyncio.sleep(0.3)
    print('setter: flipping ready ON')
    ready.set()                         # wakes up everyone waiting

await asyncio.gather(waiter(), setter())


---


## 8 - Safe Shared State: `async with` Pools + `asyncio.Lock`

Opening a DB connection is expensive (~50ms). A **pool** opens a few once and reuses them. Two async tools make this safe:
- `async with pool.acquire() as conn:` - borrow + auto-return (Day 04's `with`, now async).
- `asyncio.Lock` - ensure the pool is created **once**, even if many requests race.

> **Project:** `tools/pg_client.py` (pool + `async with`), `sql_agent/a2a_server.py` `_get_pool()` (double-checked `asyncio.Lock`).


In [ ]:
class _PooledConn:
    async def __aenter__(self):  return self
    async def __aexit__(self, *exc):  return False
    async def fetch(self, sql):
        await asyncio.sleep(0.1)
        return [{'category': 'Electronics', 'revenue': 1240500}]

class FakePool:
    def acquire(self):
        return _PooledConn()       # async context manager

_pool = None
_lock = asyncio.Lock()

async def get_pool():
    """Lazy singleton with double-checked locking (the sql_agent pattern)."""
    global _pool
    if _pool is None:                  # 1st check - fast, no lock
        async with _lock:              # only one coroutine enters at a time
            if _pool is None:          # 2nd check - another may have built it
                print('Creating pool (happens once)...')
                _pool = FakePool()
    return _pool


In [ ]:
async def run_query(sql):
    pool = await get_pool()
    async with pool.acquire() as conn:     # borrow + auto-release
        return await conn.fetch(sql)

results = await asyncio.gather(
    run_query('SELECT ...'), run_query('SELECT ...'), run_query('SELECT ...'),
)
print('All queries done. First row:', results[0])


"Creating pool" printed **once**, even though three queries raced into `get_pool()` together - that's what the double-checked `asyncio.Lock` guarantees.


---


## 9 - Going Further (know these exist; don't reach for them yet)

Real tools you'll meet eventually. **Not** in your project today, so we only name them.

| Tool | What it's for | When you'd add it |
|------|---------------|-------------------|
| `asyncio.Semaphore` | cap how many coroutines run at once | rate-limiting LLM/API calls ("max 5 in flight") |
| `asyncio.as_completed` | results **as they finish**, not in order | show fastest agent's reply first |
| `task.cancel()` / `CancelledError` | stop a task early | user closes the tab mid-stream |
| `asyncio.wait` | lower-level than gather | racing two sources, take whichever wins |


In [ ]:
# Semaphore preview - where you'd add it to cap concurrency. (Not in the project today.)
limit = asyncio.Semaphore(2)            # at most 2 at a time

async def limited_call(n):
    async with limit:                   # waits here if 2 are already running
        await asyncio.sleep(0.2)
        return f'call {n} done'

out = await asyncio.gather(*[limited_call(i) for i in range(5)])
print(out)   # all 5 complete, but never more than 2 ran at once


---


## Summary - Day 09

| Tool | Does | Project file |
|------|------|-------------|
| `async def` / `await` | define + run coroutines | every agent + route |
| `asyncio.run(main())` | entry point in a `.py` script | every agent `__main__` |
| `asyncio.gather(*tasks)` | run concurrently, results in order | `agent_health.py` |
| `asyncio.create_task()` | background, don't wait | `chat.py` saves |
| `asyncio.wait_for(c, timeout=)` | bound a slow call | `title_generator.py` |
| `return_exceptions=True` | failures as values, batch survives | `research_agent.py` |
| `asyncio.to_thread()` | blocking code off the loop | DynamoDB (x62) |
| `async def` + `yield` / `async for` | stream values over time | `supervisor_client.py` |
| `asyncio.Queue` (put/get) | producer -> consumer hand-off | `EventQueue` in every `execute()` |
| `asyncio.Event` | one-shot signal | task status flips |
| `async with pool.acquire()` | borrow + auto-return | `pg_client.py` |
| `asyncio.Lock` | safe one-time init | `sql_agent` pool |

**Four rules to remember:**
- Async helps **I/O-bound** waiting, not CPU-bound math.
- `await` in sequence is *not* concurrent - use `gather` or `create_task` to overlap.
- Never call blocking code directly in a coroutine - wrap it in `to_thread`.
- A Queue connects a producer and consumer running concurrently - that's the A2A `EventQueue`.

**Next:** Day 10 - FastAPI: turn these async functions (and that `yield` stream) into a real streaming web API.
